Start by importing the required packages.

In [64]:
import re
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

Below are dictionaries for name conversions that happen later on in the code.

In [66]:
IOC_TO_COUNTRY = {
    "AFG": "Afghanistan",           "ALB": "Albania",
    "ALG": "Algeria",               "AND": "Andorra",
    "ANG": "Angola",                "ANT": "Antigua and Barbuda",
    "ARG": "Argentina",             "ARM": "Armenia",
    "ARU": "Aruba",                 "ASA": "American Samoa",
    "AUS": "Australia",             "AUT": "Austria",
    "AZE": "Azerbaijan",            "BAH": "Bahamas",
    "BAN": "Bangladesh",            "BAR": "Barbados",
    "BDI": "Burundi",               "BEL": "Belgium",
    "BEN": "Benin",                 "BER": "Bermuda",
    "BHU": "Bhutan",                "BIH": "Bosnia and Herzegovina",
    "BIZ": "Belize",                "BLR": "Belarus",
    "BOL": "Bolivia",               "BOT": "Botswana",
    "BRA": "Brazil",                "BRN": "Bahrain",
    "BRU": "Brunei",                "BUL": "Bulgaria",
    "BUR": "Burkina Faso",          "CAF": "Central African Republic",
    "CAM": "Cambodia",              "CAN": "Canada",
    "CAY": "Cayman Islands",        "CGO": "Republic of the Congo",
    "CHA": "Chad",                  "CHI": "Chile",
    "CHN": "China",                 "CIV": "Côte d'Ivoire",
    "CMR": "Cameroon",              "COD": "Democratic Republic of the Congo",
    "COK": "Cook Islands",          "COL": "Colombia",
    "COM": "Comoros",               "CPV": "Cape Verde",
    "CRC": "Costa Rica",            "CRO": "Croatia",
    "CUB": "Cuba",                  "CYP": "Cyprus",
    "CZE": "Czech Republic",        "DEN": "Denmark",
    "DJI": "Djibouti",              "DMA": "Dominica",
    "DOM": "Dominican Republic",    "ECU": "Ecuador",
    "EGY": "Egypt",                 "ESA": "El Salvador",
    "ESP": "Spain",                 "EST": "Estonia",
    "ETH": "Ethiopia",              "FIJ": "Fiji",
    "FIN": "Finland",               "FRA": "France",
    "FSM": "Micronesia",            "GAB": "Gabon",
    "GAM": "Gambia",                "GBR": "United Kingdom",
    "GBS": "Guinea-Bissau",         "GEO": "Georgia",
    "GER": "Germany",               "GHA": "Ghana",
    "GRE": "Greece",                "GRN": "Grenada",
    "GUA": "Guatemala",             "GUI": "Guinea",
    "GUM": "Guam",                  "GUY": "Guyana",
    "HAI": "Haiti",                 "HKG": "Hong Kong",
    "HON": "Honduras",              "HUN": "Hungary",
    "INA": "Indonesia",             "IND": "India",
    "IRI": "Iran",                  "IRL": "Ireland",
    "ISL": "Iceland",               "ISR": "Israel",
    "ISV": "Virgin Islands",        "ITA": "Italy",
    "JAM": "Jamaica",               "JOR": "Jordan",
    "JPN": "Japan",                 "KAZ": "Kazakhstan",
    "KEN": "Kenya",                 "KGZ": "Kyrgyzstan",
    "KIR": "Kiribati",              "KOR": "South Korea",
    "KOS": "Kosovo",                "KSA": "Saudi Arabia",
    "KUW": "Kuwait",                "LAO": "Laos",
    "LAT": "Latvia",                "LBA": "Libya",
    "LBN": "Lebanon",               "LBR": "Liberia",
    "LCA": "Saint Lucia",           "LES": "Lesotho",
    "LIE": "Liechtenstein",         "LTU": "Lithuania",
    "LUX": "Luxembourg",            "MAD": "Madagascar",
    "MAR": "Morocco",               "MAS": "Malaysia",
    "MAW": "Malawi",                "MDA": "Moldova",
    "MDV": "Maldives",              "MEX": "Mexico",
    "MGL": "Mongolia",              "MHL": "Marshall Islands",
    "MKD": "North Macedonia",       "MLI": "Mali",
    "MLT": "Malta",                 "MON": "Monaco",
    "MOZ": "Mozambique",            "MRI": "Mauritius",
    "MTN": "Mauritania",            "MYA": "Myanmar",
    "NAM": "Namibia",               "NCA": "Nicaragua",
    "NED": "Netherlands",           "NEP": "Nepal",
    "NGR": "Nigeria",               "NIG": "Niger",
    "NOR": "Norway",                "NRU": "Nauru",
    "NZL": "New Zealand",           "OMA": "Oman",
    "PAK": "Pakistan",              "PAN": "Panama",
    "PAR": "Paraguay",              "PER": "Peru",
    "PHI": "Philippines",           "PLE": "Palestine",
    "PLW": "Palau",                 "PNG": "Papua New Guinea",
    "POL": "Poland",                "POR": "Portugal",
    "PRK": "North Korea",           "PUR": "Puerto Rico",
    "QAT": "Qatar",                 "ROU": "Romania",
    "RSA": "South Africa",          "RUS": "Russia",
    "RWA": "Rwanda",                "SAM": "Samoa",
    "SEN": "Senegal",               "SEY": "Seychelles",
    "SGP": "Singapore",             "SKN": "Saint Kitts and Nevis",
    "SLE": "Sierra Leone",          "SLO": "Slovenia",
    "SMR": "San Marino",            "SOL": "Solomon Islands",
    "SOM": "Somalia",               "SRB": "Serbia",
    "SRI": "Sri Lanka",             "SSD": "South Sudan",
    "STP": "São Tomé and Príncipe", "SUD": "Sudan",
    "SUI": "Switzerland",           "SUR": "Suriname",
    "SVK": "Slovakia",              "SWE": "Sweden",
    "SWZ": "Eswatini",              "SYR": "Syria",
    "TAN": "Tanzania",              "TGA": "Tonga",
    "THA": "Thailand",              "TJK": "Tajikistan",
    "TKM": "Turkmenistan",          "TLS": "Timor-Leste",
    "TOG": "Togo",                  "TPE": "Taiwan",
    "TTO": "Trinidad and Tobago",   "TUN": "Tunisia",
    "TUR": "Turkey",                "TUV": "Tuvalu",
    "UAE": "United Arab Emirates",  "UGA": "Uganda",
    "UKR": "Ukraine",               "URU": "Uruguay",
    "USA": "United States",         "UZB": "Uzbekistan",
    "VAN": "Vanuatu",               "VEN": "Venezuela",
    "VIE": "Vietnam",               "VIN": "Saint Vincent and the Grenadines",
    "YEM": "Yemen",                 "ZAM": "Zambia",
    "ZIM": "Zimbabwe",       
}

LOCATION_CORRECTIONS = {
    "EschAlzette, Luxembourg":  "Esch-sur-Alzette, Luxembourg",
    "Mungia-Laukariz, Spain":   "Mungia, Spain",
    "Qian Daohu, China":        "Qiandaohu, China",
    "RisskovAarhus, Denmark":   "Aarhus, Denmark",
    "Sharm ElSheikh, Egypt":    "Sharm el-Sheikh, Egypt",
}

CHALL_NAME_FIXES = {
    "Cordoba":        "Cordoba, Argentina",
    "Santiago":       "Santiago, Chile",
    "Los Cabos":      "Los Cabos, Mexico",
    "Cali":           "Cali, Colombia",
    "Lima":           "Lima, Peru",
    "Bogota":         "Bogota, Colombia",
    "Istanbul TTF":   "Istanbul",
    "Tigre":          "Tigre, Argentina",
    "Košice":         "Košice, Slovakia",
    "Villa Maria":    "Villa Maria, Argentina",
    "Montemar":       "Alicante (Montemar), Spain",
    "Mérida":         "Mérida, Mexico",
    "Brasilia":       "Brasília (Federal District), Brazil",
    "Tunis":          "Tunis city, Tunisia",
    "Salinas":        "Salinas, Ecuador",
    "Fairfield":      "Fairfield CA, United States",
    "Sumter":         "Sumter SC, United States",
    "Leon":           "Leon, Mexico",
}

Load in a CSV file and look and the column names and datatypes.

In [68]:
atp2023 = pd.read_csv('atp_matches_2023.csv')
atp2023.dtypes

tourney_id             object
tourney_name           object
surface                object
draw_size               int64
tourney_level          object
tourney_date            int64
match_num               int64
winner_id               int64
winner_seed           float64
winner_entry           object
winner_name            object
winner_hand            object
winner_ht             float64
winner_ioc             object
winner_age            float64
loser_id                int64
loser_seed            float64
loser_entry            object
loser_name             object
loser_hand             object
loser_ht              float64
loser_ioc              object
loser_age             float64
score                  object
best_of                 int64
round                  object
minutes               float64
w_ace                 float64
w_df                  float64
w_svpt                float64
w_1stIn               float64
w_1stWon              float64
w_2ndWon              float64
w_SvGms   

Drop unnecessary columns, change the date column to datetime, and list unique values in the tourney_name column.

In [70]:
atp2023 = atp2023[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]
atp2023.dtypes

tourney_id       object
tourney_name     object
surface          object
draw_size         int64
tourney_level    object
tourney_date      int64
dtype: object

In [71]:
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals', 'Davis Cup

Remove the values that contain "Davis Cup". These won't be used in the analysis.

In [73]:
atp2023 = atp2023[~atp2023['tourney_name'].str.contains('Davis Cup', case=False, na=False)]
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=obj

Remove "Masters" from any of the values. This will help find the locations.

In [75]:
atp2023["tourney_name"] = atp2023["tourney_name"].str.replace("masters", "", case=False, regex=True).str.strip()
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells', 'Miami', 'Estoril', 'Houston', 'Marrakech',
       'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka', 'Madrid',
       'Rome', 'Geneva', 'Lyon', 'Roland Garros', 's Hertogenbosch',
       'Stuttgart', 'Halle', "Queen's Club", 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos', 'Washington', 'Canada',
       'Cincinnati', 'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=object)

Change the tournament names without clear locations to be more specific.

In [77]:
atp2023["tourney_name"] = atp2023["tourney_name"].replace({
    "Australian Open": "Melbourne",
    "Roland Garros": "Paris",
    "Queen's Club": "London",
    "Canada": "Toronto",
    "Us Open": "New York",
    "Tour Finals": "Turin",
    "NextGen Finals": "Jeddah",
    "Cordoba": "Cordoba, Argentina",
    "Los Cabos": "Los Cabos, Mexico",
    "Santiago": "Santiago, Chile",
})
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Melbourne', 'Cordoba, Argentina', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai',
       'Santiago, Chile', 'Indian Wells', 'Miami', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka',
       'Madrid', 'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch',
       'Stuttgart', 'Halle', 'London', 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos, Mexico', 'Washington', 'Toronto',
       'Cincinnati', 'Winston-Salem', 'New York', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin', 'Jeddah'],
      dtype=object)

Change "United Cup" to the locations where the event is held.

In [79]:
atp2023["tourney_name"] = atp2023["tourney_name"].apply(
    lambda x: ["Brisbane", "Sydney", "Perth"] if x == "United Cup" else [x]
)

atp2023 = atp2023.explode("tourney_name")
atp2023["tourney_name"].unique()

array(['Brisbane', 'Sydney', 'Perth', 'Adelaide 1', 'Pune', 'Auckland',
       'Adelaide 2', 'Melbourne', 'Cordoba, Argentina', 'Dallas',
       'Montpellier', 'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai',
       'Santiago, Chile', 'Indian Wells', 'Miami', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka',
       'Madrid', 'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch',
       'Stuttgart', 'Halle', 'London', 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos, Mexico', 'Washington', 'Toronto',
       'Cincinnati', 'Winston-Salem', 'New York', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin', 'Jeddah'],
      dtype=object)

Add a year column for this file (2023)

In [81]:
year = 2023
atp2023["year"] = year
atp2023 

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,year
0,2023-9900,Brisbane,Hard,18,A,20230102,2023
0,2023-9900,Sydney,Hard,18,A,20230102,2023
0,2023-9900,Perth,Hard,18,A,20230102,2023
1,2023-9900,Brisbane,Hard,18,A,20230102,2023
1,2023-9900,Sydney,Hard,18,A,20230102,2023
...,...,...,...,...,...,...,...
2761,2023-7696,Jeddah,Hard,8,F,20231127,2023
2762,2023-7696,Jeddah,Hard,8,F,20231127,2023
2763,2023-7696,Jeddah,Hard,8,F,20231127,2023
2764,2023-7696,Jeddah,Hard,8,F,20231127,2023


Drop duplicate tournament names so you have 1 row for each unique location

In [83]:
atp2023 = atp2023.drop_duplicates(subset=["tourney_id", "tourney_name"]).reset_index(drop=True)
atp2023

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,year
0,2023-9900,Brisbane,Hard,18,A,20230102,2023
1,2023-9900,Sydney,Hard,18,A,20230102,2023
2,2023-9900,Perth,Hard,18,A,20230102,2023
3,2023-2843,Adelaide 1,Hard,32,A,20230102,2023
4,2023-0891,Pune,Hard,32,A,20230102,2023
...,...,...,...,...,...,...,...
63,2023-0337,Vienna,Hard,32,A,20231023,2023
64,2023-0341,Metz,Hard,32,A,20231106,2023
65,2023-7434,Sofia,Hard,32,A,20231106,2023
66,2023-0605,Turin,Hard,8,A,20231113,2023


Create a dataframe that contains a list of the different locations tournaments are held and the locations for each one.

In [85]:
geolocator = Nominatim(user_agent="Professional_Men's_Tennis2023-2026_Analysis")  # The file the function is being run on
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)                    # Nominatim rate limit

unique_cities = atp2023["tourney_name"]                                           # Create a list of all the cities

location_data = {}                                                                # Create dictionary
for city in unique_cities:                                                        # Run for loop with geocode function to locate each city 
    location = geocode(city)
    if location:                                                                  # If else logic to find latitude and longitude of each city and place in dictionary
        location_data[city] = (location.latitude, location.longitude)
    else:
        print(f"Could not geocode: {city}")                                       # Flag any misses
location_data
coord_df = pd.DataFrame.from_dict(                                                # Create df from dictionary populated above 
    location_data, orient="index", columns=["latitude", "longitude"]
)
coord_df.index.name = "tourney_name"                                              # Make the index the tournament name/city and reset the index
coord_df = coord_df.reset_index()
coord_df

,tourney_name,latitude,longitude
0,Brisbane,-27.468962,153.023501
1,Sydney,-33.869844,151.208285
2,Perth,-31.955897,115.860578
3,Adelaide 1,-34.819459,138.504749
4,Pune,18.521374,73.854507
...,...,...,...
63,Vienna,48.208354,16.372504
64,Metz,49.119696,6.176355
65,Sofia,42.697703,23.321736
66,Turin,45.067755,7.682489


In [86]:
atp2023 = atp2023.merge(coord_df, on="tourney_name", how="left")
atp2023

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,year,latitude,longitude
0,2023-9900,Brisbane,Hard,18,A,20230102,2023,-27.468962,153.023501
1,2023-9900,Sydney,Hard,18,A,20230102,2023,-33.869844,151.208285
2,2023-9900,Perth,Hard,18,A,20230102,2023,-31.955897,115.860578
3,2023-2843,Adelaide 1,Hard,32,A,20230102,2023,-34.819459,138.504749
4,2023-0891,Pune,Hard,32,A,20230102,2023,18.521374,73.854507
...,...,...,...,...,...,...,...,...,...
63,2023-0337,Vienna,Hard,32,A,20231023,2023,48.208354,16.372504
64,2023-0341,Metz,Hard,32,A,20231106,2023,49.119696,6.176355
65,2023-7434,Sofia,Hard,32,A,20231106,2023,42.697703,23.321736
66,2023-0605,Turin,Hard,8,A,20231113,2023,45.067755,7.682489


In [24]:
coord_df

,tourney_name,latitude,longitude,tourney_date
0,Brisbane,-27.468962,153.023501,20230102
1,Sydney,-33.869844,151.208285,20230102
2,Perth,-31.955897,115.860578,20230102
3,Adelaide 1,-34.819459,138.504749,20230102
4,Pune,18.521374,73.854507,20230102
...,...,...,...,...
63,Vienna,48.208354,16.372504,20231023
64,Metz,49.119696,6.176355,20231106
65,Sofia,42.697703,23.321736,20231106
66,Turin,45.067755,7.682489,20231113


In [25]:
coord_df["tourney_date"] = pd.to_datetime(
    coord_df["tourney_date"].astype(str),
    format="mixed"
)
coord_df["tourney_date"] = coord_df["tourney_date"].dt.year
coord_df.dtypes

tourney_name     object
latitude        float64
longitude       float64
tourney_date      int32
dtype: object

In [26]:
coord_df

,tourney_name,latitude,longitude,tourney_date
0,Brisbane,-27.468962,153.023501,2023
1,Sydney,-33.869844,151.208285,2023
2,Perth,-31.955897,115.860578,2023
3,Adelaide 1,-34.819459,138.504749,2023
4,Pune,18.521374,73.854507,2023
...,...,...,...,...
63,Vienna,48.208354,16.372504,2023
64,Metz,49.119696,6.176355,2023
65,Sofia,42.697703,23.321736,2023
66,Turin,45.067755,7.682489,2023


In [27]:
def clean_atp(year):
    df = pd.read_csv(f"atp_matches_{year}.csv")
    df = df[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]
    
    df = df[~df['tourney_name'].str.contains('Davis Cup|Laver Cup', case=False, na=False)]
    df["tourney_name"] = df["tourney_name"].str.replace("masters", "", case=False, regex=True).str.strip()
    
    df["tourney_name"] = df["tourney_name"].replace({
        "Australian Open": "Melbourne",
        "Roland Garros": "Paris",
        "Queen's Club": "London",
        "Canada": "Montreal" if year % 2 == 0 else "Toronto",
        "Us Open": "New York",
        "Tour Finals": "Turin",
        "NextGen Finals": "Jeddah",
        "Next Gen Finals": "Jeddah",
        "Cordoba": "Cordoba, Argentina",
        "Los Cabos": "Los Cabos, Mexico",
        "Santiago": "Santiago, Chile",
        "Paris Olympics": "Paris"
    })
    
    df["tourney_name"] = df["tourney_name"].apply(
        lambda x: ["Brisbane", "Sydney", "Perth"] if x == "United Cup" and year == 2023
        else ["Sydney", "Perth"] if x == "United Cup"
        else [x]
    )
    df = df.explode("tourney_name")
    
    df["year"] = year
    df = df.drop_duplicates(subset=["tourney_name", "year"])
    return df

In [28]:
clean_atp(2024)

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,year
0,2024-0339,Brisbane,Hard,32,A,20240101,2024
31,2024-9900,Sydney,Hard,18,A,20240101,2024
31,2024-9900,Perth,Hard,18,A,20240101,2024
56,2024-0336,Hong Kong,Hard,32,A,20240101,2024
83,2024-8998,Adelaide,Hard,32,A,20240108,2024
...,...,...,...,...,...,...,...
2646,2024-0337,Vienna,Hard,32,A,20241021,2024
2732,2024-4787,Belgrade,Hard,28,A,20241104,2024
2759,2024-0341,Metz,Hard,28,A,20241104,2024
2786,2024-0605,Turin,Hard,8,F,20241111,2024


In [29]:
def parse_futures_name(name: str) -> str:
    name = re.sub(r'^M\d+(\+H)?\s+', '', name)  # remove M15, M25, M15+H, M25+H prefix
    name = re.sub(r'\s*\+H$', '', name)           # remove trailing +H (2026 format)
    name = re.sub(r'\s*\(.*?\)', '', name)         # remove anything in parentheses e.g. (moved from Coquimbo)
    return name.strip()

In [30]:
def clean_atp_futures(year):
    df = pd.read_csv(f"atp_matches_futures_{year}.csv")
    df = df[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]

    df["tourney_name"] = df["tourney_name"].apply(parse_futures_name)

    df["country_code"] = df["tourney_id"].str.split("-").str[3]
    df["tourney_name"] = df["tourney_name"] + ", " + df["country_code"].map(IOC_TO_COUNTRY).fillna(df["country_code"])
    df = df.drop(columns=["country_code"])

    df["year"] = year
    df = df.drop_duplicates(subset=["tourney_name", "year"])
    return df

In [31]:
clean_atp_futures(2023)

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,year
0,2023-M-ITF-ARG-01A-2023,"Tucuman, Argentina",Clay,32,15,20230220,2023
62,2023-M-ITF-ARG-03A-2023,"Rio Cuarto, Argentina",Clay,32,25,20230306,2023
93,2023-M-ITF-ARG-04A-2023,"Rosario-Santa Fe, Argentina",Clay,32,25,20230626,2023
124,2023-M-ITF-ARG-05A-2023,"Olavarria, Argentina",Clay,32,15,20230904,2023
155,2023-M-ITF-ARG-06A-2023,"Buenos Aires, Argentina",Clay,32,15,20230828,2023
...,...,...,...,...,...,...,...
17471,2023-M-ITF-ROU-05A-2023,"Brasov, Romania",Clay,32,25,20230703,2023
17502,2023-M-ITF-LUX-02A-2023,"Esch/Alzette, Luxembourg",Clay,32,25,20230710,2023
17533,2023-M-ITF-CAN-03A-2023,"Laval, Canada",Hard,32,25,20230710,2023
17564,2023-M-ITF-ROU-06A-2023,"Bacau, Romania",Clay,32,25,20230724,2023


In [32]:
def clean_atp_chall(year):
    df = pd.read_csv(f"atp_matches_qual_chall_{year}.csv")
    df = df[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]

    # Keep challenger events only
    df = df[df["tourney_level"] == "C"].copy()

    # Remove "CH" as a standalone token, then fix ambiguous city names
    df["tourney_name"] = (
        df["tourney_name"]
        .str.replace(r"\bCH\b", "", regex=True)
        .str.strip()
        .replace(CHALL_NAME_FIXES)
    )

    df["year"] = year
    df = df.drop_duplicates(subset=["tourney_name", "year"])
    return df

In [33]:
clean_atp_chall(2023)

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,year
0,2023-0059,"Salinas, Ecuador",Hard,32,C,20230724,2023
45,2023-0213,San Luis Potosi,Clay,32,C,20230403,2023
91,2023-0221,Tampere,Clay,32,C,20230717,2023
328,2023-0363,Knoxville,Hard,32,C,20231106,2023
389,2023-0398,Cherbourg,Hard,32,C,20230213,2023
...,...,...,...,...,...,...,...
10424,2023-9695,Tigre 2,Clay,32,C,20230109,2023
10471,2023-9697,Mauthausen,Clay,32,C,20230508,2023
10517,2023-9814,"Cali, Colombia",Clay,32,C,20230619,2023
10565,2023-9817,Roseto Degli Abruzzi,Clay,32,C,20230416,2023


In [34]:
years = [2023, 2024, 2025, 2026]

atp_df     = pd.concat([clean_atp(y)         for y in years], ignore_index=True)
futures_df = pd.concat([clean_atp_futures(y) for y in years], ignore_index=True)
chall_df   = pd.concat([clean_atp_chall(y)   for y in years], ignore_index=True)

all_tournaments = pd.concat([atp_df, futures_df, chall_df], ignore_index=True)
all_tournaments = all_tournaments.drop_duplicates(subset=["tourney_name", "year"])
all_tournaments["tourney_name"] = all_tournaments["tourney_name"].replace(LOCATION_CORRECTIONS)

In [35]:
all_tournaments

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,year
0,2023-9900,Brisbane,Hard,18,A,20230102,2023
1,2023-9900,Sydney,Hard,18,A,20230102,2023
2,2023-9900,Perth,Hard,18,A,20230102,2023
3,2023-2843,Adelaide 1,Hard,32,A,20230102,2023
4,2023-0891,Pune,Hard,32,A,20230102,2023
...,...,...,...,...,...,...,...
2072,2026-3089,Centurion 2,Hard,32,C,20260601,2026
2073,2026-0460,Heilbronn,Clay,32,C,20260601,2026
2074,2026-9001,Perugia,Clay,32,C,20260601,2026
2075,2026-0558,Prostejov,Clay,32,C,20260601,2026


In [41]:
geolocator = Nominatim(user_agent="Professional_Men's_Tennis2023-2026_Analysis")
geocode    = RateLimiter(geolocator.geocode, min_delay_seconds=1)

unique_cities = all_tournaments["tourney_name"]

location_data = {}
for city in unique_cities:
    loc = geocode(city)
    if loc:
        location_data[city] = (loc.latitude, loc.longitude)
    else:
        print(f"Could not geocode: {city}")

new_df = pd.DataFrame.from_dict(
    location_data, orient="index", columns=["latitude", "longitude"]
)
new_df.index.name = "tourney_name"
new_df = new_df.reset_index()

RateLimiter caught an error, retrying (0/2 tries). Called with (*('Lakewood CA, United States',), **{}).
Traceback (most recent call last):
  File "C:\Users\Utlisateur\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 536, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "C:\Users\Utlisateur\anaconda3\Lib\site-packages\urllib3\connection.py", line 464, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Utlisateur\anaconda3\Lib\http\client.py", line 1428, in getresponse
    response.begin()
  File "C:\Users\Utlisateur\anaconda3\Lib\http\client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Utlisateur\anaconda3\Lib\http\client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:

In [42]:
final_df = all_tournaments.merge(
    new_df[["tourney_name", "latitude", "longitude"]],
    on="tourney_name",
    how="left",
)

final_df.to_csv("atp_all_locations.csv", index=False)

In [43]:
final_df

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,year,latitude,longitude
0,2023-9900,Brisbane,Hard,18,A,20230102,2023,-27.468962,153.023501
1,2023-9900,Sydney,Hard,18,A,20230102,2023,-33.869844,151.208285
2,2023-9900,Perth,Hard,18,A,20230102,2023,-31.955897,115.860578
3,2023-2843,Adelaide 1,Hard,32,A,20230102,2023,-34.819459,138.504749
4,2023-0891,Pune,Hard,32,A,20230102,2023,18.521374,73.854507
...,...,...,...,...,...,...,...,...,...
2027,2026-3089,Centurion 2,Hard,32,C,20260601,2026,39.849914,-89.630034
2028,2026-0460,Heilbronn,Clay,32,C,20260601,2026,49.142291,9.218655
2029,2026-9001,Perugia,Clay,32,C,20260601,2026,43.107032,12.402996
2030,2026-0558,Prostejov,Clay,32,C,20260601,2026,49.472147,17.111798
